# Week 9 — Retrieval-Ready Study Assistant
## PariShiksha | NCERT Class 9 Science
### Chapter 8: Force and Laws of Motion

**Student:** PG Diploma AI-ML | Cohort 2  
**Chapter:** iesc108.pdf — Force and Laws of Motion  
**Pipeline:** PDF Extraction → Tokenizer Comparison → BERT Chunking → BM25 Retrieval → Groq Generation → Evaluation

## Stage 1 — PDF Extraction
Extracting raw text from NCERT Chapter 8 PDF using PyMuPDF.
Printing total characters to verify extraction worked correctly.

In [2]:
import fitz  # pymupdf
import re
import sys

print(f"Python: {sys.executable}")
print(f"PyMuPDF version: {fitz.__version__}")

# load PDF — update path to your local file
doc = fitz.open(r"iesc108.pdf")

# extract all pages
full_text = ""
for page in doc:
    full_text += page.get_text()

print(f"\nTotal pages: {len(doc)}")
print(f"Total characters: {len(full_text)}")
print("\n--- FIRST 2000 CHARS ---")
print(full_text[:2000])

Python: c:\Users\siddp\Downloads\week9-parishiksha\venv\Scripts\python.exe
PyMuPDF version: 1.27.2.2

Total pages: 13
Total characters: 33890

--- FIRST 2000 CHARS ---
In the previous chapter, we described the
motion of an object along a straight line in
terms of its position, velocity and acceleration.
We saw that such a motion can be uniform
or non-uniform. We have not yet discovered
what causes the motion. Why does the speed
of an object change with time? Do all motions
require a cause? If so, what is the nature of
this cause? In this chapter we shall make an
attempt to quench all such curiosities.
For many centuries, the problem of
motion and its causes had puzzled scientists
and philosophers. A ball on the ground, when
given a small hit, does not move forever. Such
observations suggest that rest is the “natural
state” of an object. This remained the belief
until Galileo Galilei and Isaac Newton
developed an entirely different approach to
understand motion.
In our everyday life we 

## Stage 1 — Sample Passages
Selecting 5 representative passages for tokenizer comparison:
concept paragraph, worked example, formula, end-of-chapter question, figure caption.

In [3]:
sample_passages = [
    # 1. Concept paragraph
    "In the previous chapter, we described the motion of an object along a straight line in terms of its position, velocity and acceleration. We saw that such a motion can be uniform or non-uniform.",

    # 2. Worked example
    "Example 8.1 A constant force acts on an object of mass 5 kg for a duration of 2 s. It increases the object's velocity from 3 m s-1 to 7 m s-1. Find the magnitude of the applied force.",

    # 3. Formula
    "The momentum p of an object is defined as the product of its mass m and velocity v. That is p = mv. Ft = mv minus mu. F = ma.",

    # 4. End of chapter question
    "6. A stone of 1 kg is thrown with a velocity of 20 m s-1 across the frozen surface of a lake and comes to rest after travelling a distance of 50 m. What is the force of friction between the stone and the ice?",

    # 5. Figure caption
    "Fig. 8.5: (a) the downward motion; (b) the upward motion of a marble on an inclined plane; and (c) on a double inclined plane."
]

print(f"Total sample passages: {len(sample_passages)}")
for i, p in enumerate(sample_passages):
    print(f"\nPassage {i+1}: {p[:80]}...")

Total sample passages: 5

Passage 1: In the previous chapter, we described the motion of an object along a straight l...

Passage 2: Example 8.1 A constant force acts on an object of mass 5 kg for a duration of 2 ...

Passage 3: The momentum p of an object is defined as the product of its mass m and velocity...

Passage 4: 6. A stone of 1 kg is thrown with a velocity of 20 m s-1 across the frozen surfa...

Passage 5: Fig. 8.5: (a) the downward motion; (b) the upward motion of a marble on an incli...


## Stage 1 — Text Preprocessing
Cleaning raw PDF text by removing page numbers, extra whitespace,
and fixing hyphenated words broken across lines.

In [4]:
def preprocess(text):
    # remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    # remove standalone page numbers
    text = re.sub(r'\n\d+\n', ' ', text)
    # fix hyphenated words broken across lines
    text = re.sub(r'-\n', '', text)
    return text.strip()

clean_text = preprocess(full_text)

print(f"Before: {len(full_text)} chars")
print(f"After:  {len(clean_text)} chars")
print(f"Removed: {len(full_text) - len(clean_text)} chars")
print("\n--- CLEAN TEXT SAMPLE ---")
print(clean_text[:2000])

Before: 33890 chars
After:  33846 chars
Removed: 44 chars

--- CLEAN TEXT SAMPLE ---
In the previous chapter, we described the motion of an object along a straight line in terms of its position, velocity and acceleration. We saw that such a motion can be uniform or non-uniform. We have not yet discovered what causes the motion. Why does the speed of an object change with time? Do all motions require a cause? If so, what is the nature of this cause? In this chapter we shall make an attempt to quench all such curiosities. For many centuries, the problem of motion and its causes had puzzled scientists and philosophers. A ball on the ground, when given a small hit, does not move forever. Such observations suggest that rest is the “natural state” of an object. This remained the belief until Galileo Galilei and Isaac Newton developed an entirely different approach to understand motion. In our everyday life we observe that some effort is required to put a stationary object into motion or to s

## Stage 1 — Tokenizer Comparison
Comparing BERT WordPiece and T5 SentencePiece on 5 representative passages.
Measuring token counts and how scientific terms get split.

In [5]:
from transformers import BertTokenizer, T5Tokenizer

bert = BertTokenizer.from_pretrained('bert-base-uncased')
t5 = T5Tokenizer.from_pretrained('t5-small')

print("=== TOKENIZER COMPARISON: BERT vs T5 ===")
print(f"{'Passage':<12} {'BERT':>6} {'T5':>6} {'Winner':>8}")
print("-" * 40)

for i, passage in enumerate(sample_passages):
    bert_tokens = bert.tokenize(passage)
    t5_tokens = t5.tokenize(passage)
    winner = "BERT" if len(bert_tokens) <= len(t5_tokens) else "T5"
    print(f"Passage {i+1:<5} {len(bert_tokens):>6} {len(t5_tokens):>6} {winner:>8}")

print("\n=== SCIENTIFIC WORD ANALYSIS ===")
test_words = ["acceleration", "velocity", "non-uniform", "momentum", "thermodynamics"]
for word in test_words:
    b = bert.tokenize(word)
    t = t5.tokenize(word)
    print(f"\n{word}:")
    print(f"  BERT ({len(b)} tokens): {b}")
    print(f"  T5   ({len(t)} tokens): {t}")

print("\n=== DECISION ===")
print("BERT WordPiece chosen because:")
print("- Fewer tokens across all 5 passages")
print("- Better handling of hyphenated scientific terms")
print("- Trained on Wikipedia = knows physics vocabulary")

c:\Users\siddp\Downloads\week9-parishiksha\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== TOKENIZER COMPARISON: BERT vs T5 ===
Passage        BERT     T5   Winner
----------------------------------------
Passage 1         40     44     BERT
Passage 2         50     53     BERT
Passage 3         35     45     BERT
Passage 4         50     56     BERT
Passage 5         37     39     BERT

=== SCIENTIFIC WORD ANALYSIS ===

acceleration:
  BERT (1 tokens): ['acceleration']
  T5   (1 tokens): ['▁acceleration']

velocity:
  BERT (1 tokens): ['velocity']
  T5   (1 tokens): ['▁velocity']

non-uniform:
  BERT (3 tokens): ['non', '-', 'uniform']
  T5   (5 tokens): ['▁non', '-', 'un', 'i', 'form']

momentum:
  BERT (1 tokens): ['momentum']
  T5   (1 tokens): ['▁momentum']

thermodynamics:
  BERT (5 tokens): ['the', '##rm', '##od', '##yna', '##mics']
  T5   (3 tokens): ['▁thermo', 'dynamic', 's']

=== DECISION ===
BERT WordPiece chosen because:
- Fewer tokens across all 5 passages
- Better handling of hyphenated scientific terms
- Trained on Wikipedia = knows physics vocabulary


## Stage 1 — Chunking Experiment (Stretch Tier)
Comparing three chunking strategies:
1. 300-word chunks (baseline)
2. 150-word chunks
3. 150 BERT token chunks (final choice)

Motivated by retrieval failures on Newton's first law query.

In [6]:
# Strategy 1: 300-word chunks
def chunk_text(text, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []
    i = 0
    chunk_id = 0
    while i < len(words):
        chunk_words = words[i:i+chunk_size]
        chunk_text_str = ' '.join(chunk_words)
        chunks.append({
            'id': chunk_id,
            'text': chunk_text_str,
            'chapter': 'Force and Laws of Motion',
            'content_type': 'general',
            'word_count': len(chunk_words)
        })
        chunk_id += 1
        i += chunk_size - overlap
    return chunks

chunks_300 = chunk_text(clean_text, chunk_size=300, overlap=50)
chunks_150 = chunk_text(clean_text, chunk_size=150, overlap=30)

print(f"300-word chunks: {len(chunks_300)}")
print(f"150-word chunks: {len(chunks_150)}")
print(f"\nSample chunk 0 (300 words):")
print(chunks_300[0]['text'][:300])

300-word chunks: 27
150-word chunks: 55

Sample chunk 0 (300 words):
In the previous chapter, we described the motion of an object along a straight line in terms of its position, velocity and acceleration. We saw that such a motion can be uniform or non-uniform. We have not yet discovered what causes the motion. Why does the speed of an object change with time? Do al


In [7]:
# Strategy 3: BERT token-based chunking with content type tagging
from collections import Counter

bert_chunker = BertTokenizer.from_pretrained('bert-base-uncased')

def chunk_by_bert_tokens(text, max_tokens=150, overlap=30):
    tokens = bert_chunker.tokenize(text)
    print(f"Total BERT tokens in chapter: {len(tokens)}")

    chunks = []
    i = 0
    chunk_id = 0

    while i < len(tokens):
        chunk_tokens = tokens[i:i+max_tokens]
        chunk_text_str = bert_chunker.convert_tokens_to_string(chunk_tokens)

        # auto content type tagging
        if any(w in chunk_text_str.lower() for w in ['example', 'solution', 'calculate']):
            content_type = 'worked_example'
        elif any(w in chunk_text_str.lower() for w in ['activity', 'observe', 'place', 'hold']):
            content_type = 'activity'
        elif any(w in chunk_text_str.lower() for w in ['fig.', 'figure', 'shows']):
            content_type = 'figure'
        elif chunk_text_str.strip() and chunk_text_str.strip()[0].isdigit():
            content_type = 'exercise_question'
        elif any(w in chunk_text_str.lower() for w in ['law', 'defined', 'states', 'tendency', 'property']):
            content_type = 'concept'
        else:
            content_type = 'general'

        chunks.append({
            'id': chunk_id,
            'text': chunk_text_str,
            'chapter': 'Force and Laws of Motion',
            'content_type': content_type,
            'word_count': len(chunk_tokens)
        })
        chunk_id += 1
        i += max_tokens - overlap

    types = Counter(c['content_type'] for c in chunks)
    print(f"Total chunks: {len(chunks)}")
    print(f"Content types: {dict(types)}")
    return chunks

chunks_bert = chunk_by_bert_tokens(clean_text)

Token indices sequence length is longer than the specified maximum sequence length for this model (8118 > 512). Running this sequence through the model will result in indexing errors


Total BERT tokens in chapter: 8118
Total chunks: 68
Content types: {'general': 10, 'activity': 16, 'figure': 14, 'concept': 12, 'worked_example': 16}


## Stage 2 — BM25 Retrieval
Building BM25 index on BERT chunks with content-type boosting.
Testing retrieval on sample queries and comparing chunk size impact.

In [8]:
from rank_bm25 import BM25Okapi

# build BM25 on 300-word chunks (baseline)
bm25_300 = BM25Okapi([c['text'].lower().split() for c in chunks_300])

# build BM25 on BERT chunks (final)
bm25_bert = BM25Okapi([c['text'].lower().split() for c in chunks_bert])

def retrieve_bert(query, top_k=3, prefer_type=None):
    tokenized_query = query.lower().split()
    scores = bm25_bert.get_scores(tokenized_query)
    results = []
    for idx, score in enumerate(scores):
        chunk = chunks_bert[idx]
        if prefer_type and chunk['content_type'] == prefer_type:
            score = score * 1.5  # boost matching content type
        results.append({'chunk': chunk, 'score': score})
    results = sorted(results, key=lambda x: x['score'], reverse=True)[:top_k]
    return results

# chunking experiment comparison
test_queries = [
    "What is Newton's first law of motion?",
    "What is the formula for force?",
    "What is inertia?"
]

print("=== CHUNKING EXPERIMENT: 300-word vs BERT chunks ===")
for q in test_queries:
    # 300-word score
    scores_300 = bm25_300.get_scores(q.lower().split())
    best_300 = max(scores_300)
    best_chunk_300 = chunks_300[scores_300.argmax() if hasattr(scores_300, 'argmax') else list(scores_300).index(max(scores_300))]

    # BERT score
    results_bert = retrieve_bert(q, prefer_type='concept')
    best_bert = results_bert[0]['score']
    best_chunk_bert = results_bert[0]['chunk']

    print(f"\nQuery: {q}")
    print(f"300-word: score={best_300:.2f} → {best_chunk_300['text'][:100]}")
    print(f"BERT:     score={best_bert:.2f} → {best_chunk_bert['text'][:100]}")

=== CHUNKING EXPERIMENT: 300-word vs BERT chunks ===

Query: What is Newton's first law of motion?
300-word: score=4.63 → a smooth marble and a smooth plane and providing a lubricant on top of the planes. Fig. 8.5: (a) the
BERT:     score=8.78 → all objects resist a change in their state of motion. in a qualitative way, the tendency of undistur

Query: What is the formula for force?
300-word: score=5.43 → the balance B on balance A. Any of these two forces can be called as action and the other as reactio
BERT:     score=6.98 → the third law of motion i. e., to every action there is an equal and opposite reaction. however, it 

Query: What is inertia?
300-word: score=3.95 → • Place a water-filled tumbler on a tray. • Hold the tray and turn around as fast as you can. • We o
BERT:     score=4.81 → . 3 • place a water - filled tumbler on a tray. • hold the tray and turn around as fast as you can. 


## Stage 3 — Groq API Connection
Connecting Groq API (llama-3.3-70b-versatile) for grounded answer generation.
Note: Gemini API had quota issues (429), switched to Groq free tier.

In [9]:
from groq import Groq

# replace with your Groq API key from console.groq.com
GROQ_API_KEY = "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"

client = Groq(api_key=GROQ_API_KEY)

# test connection
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say hello in one sentence"}]
)
print("Groq connected:", response.choices[0].message.content)

Groq connected: Hello, it's nice to meet you and I'm here to help answer any questions you might have.


## Stage 3 — Grounding Prompt + Answer Function
Building grounded answer function that only answers from retrieved NCERT chunks.
Refuses when answer not found in context (out-of-scope handling).

**v1 prompt:** permissive — 'answer only from context'  
**Final prompt:** strict numbered rules — 'do NOT use own knowledge under any circumstances'

In [10]:
def answer(question, top_k=3):
    # Step 1: retrieve relevant chunks using BERT BM25
    results = retrieve_bert(question, top_k=top_k, prefer_type='concept')

    # Step 2: build context from top chunks
    context = ""
    for i, r in enumerate(results):
        context += f"Chunk {i+1}:\n{r['chunk']['text']}\n\n"

    # Step 3: grounding prompt (final version)
    prompt = f"""You are a strict study assistant for NCERT Class 9 Science.

STRICT RULES:
1. You must ONLY use information from the CONTEXT below.
2. If the exact answer is not in the CONTEXT, say exactly:
   "I cannot find this in the provided chapter content."
3. Do NOT use your own knowledge under any circumstances.
4. Do NOT add any information not present in CONTEXT.
5. Quote directly from context where possible.

CONTEXT:
{context}

QUESTION: {question}

ANSWER (only from context above):"""

    # Step 4: call Groq with temperature=0 for reproducibility
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    answer_text = response.choices[0].message.content

    return {
        'question': question,
        'answer': answer_text,
        'retrieved_chunks': [r['chunk']['text'][:150] for r in results]
    }

# manual test on 4 questions
test_questions = [
    "What is Newton's first law of motion?",
    "What is inertia?",
    "What is the formula for force?",
    "Who is the Prime Minister of India?"  # out of scope
]

for q in test_questions:
    print(f"\nQuestion: {q}")
    result = answer(q)
    print(f"Answer: {result['answer'][:200]}")
    print(f"Retrieved chunk: {result['retrieved_chunks'][0][:100]}")
    print("-" * 50)


Question: What is Newton's first law of motion?
Answer: The first law of motion is also known as the law of inertia. "all objects resist a change in their state of motion. in a qualitative way, the tendency of undisturbed objects to stay at rest or to keep
Retrieved chunk: all objects resist a change in their state of motion. in a qualitative way, the tendency of undistur
--------------------------------------------------

Question: What is inertia?
Answer: "This property of an object is called its inertia. ... if it is at rest it tends to remain at rest ; if it is moving it tends to keep moving."
Retrieved chunk: . 3 • place a water - filled tumbler on a tray. • hold the tray and turn around as fast as you can. 
--------------------------------------------------

Question: What is the formula for force?
Answer: The formula for force is given as: f = k m a (8.3), where a = (v - u) / t, and k is a constant of proportionality. Alternatively, it is also given as: f = (m × (v - u)) / t.
R

## Stage 4 — Evaluation
Testing 17 questions on 3 axes:
- **Correctness**: correct / partial / wrong
- **Grounding**: is answer supported by retrieved chunk?
- **Refusal**: did system refuse out-of-scope questions?

Eval set: 10 direct, 3 paraphrased, 4 out-of-scope.

In [11]:
eval_questions = [
    # Direct from textbook (10)
    "What is Newton's first law of motion?",
    "What is inertia?",
    "What is Newton's second law of motion?",
    "What is momentum?",
    "What is Newton's third law of motion?",
    "What happens when an unbalanced force acts on an object?",
    "What is the SI unit of force?",
    "What is the relationship between force mass and acceleration?",
    "What is the law of conservation of momentum?",
    "What happens to a body at rest when no force acts on it?",
    # Paraphrased (3)
    "If no external force acts on a moving object what happens?",
    "How does mass affect the motion of an object?",
    "Why do passengers fall forward when a bus brakes suddenly?",
    # Out of scope (4)
    "Who is the Prime Minister of India?",
    "What is photosynthesis?",
    "What is the capital of France?",
    "Explain quantum entanglement from this chapter."
]

# run all questions
for q in eval_questions:
    result = answer(q)
    print(f"\nQ: {q}")
    print(f"A: {result['answer'][:200]}")
    print(f"Chunk: {result['retrieved_chunks'][0][:100]}")
    print("-" * 50)


Q: What is Newton's first law of motion?
A: The first law of motion is also known as the law of inertia. "all objects resist a change in their state of motion. in a qualitative way, the tendency of undisturbed objects to stay at rest or to keep
Chunk: all objects resist a change in their state of motion. in a qualitative way, the tendency of undistur
--------------------------------------------------

Q: What is inertia?
A: "This property of an object is called its inertia. ... if it is at rest it tends to remain at rest ; if it is moving it tends to keep moving."
Chunk: . 3 • place a water - filled tumbler on a tray. • hold the tray and turn around as fast as you can. 
--------------------------------------------------

Q: What is Newton's second law of motion?
A: The second law of motion states that "the rate of change of momentum of an object is proportional to the applied unbalanced force in the direction of force."
Chunk: one or two persons give a sudden push ( unbalanced force )

In [12]:
import csv

eval_scored = [
    {"question": "What is Newton's first law of motion?",
     "answer_short": "Law of inertia - objects resist change in motion",
     "correctness": "correct", "grounded": "yes", "refusal": "na"},
    {"question": "What is inertia?",
     "answer_short": "Property to resist change in state of motion",
     "correctness": "correct", "grounded": "no", "refusal": "na"},
    {"question": "What is Newton's second law of motion?",
     "answer_short": "Rate of change of momentum proportional to force",
     "correctness": "correct", "grounded": "no", "refusal": "na"},
    {"question": "What is momentum?",
     "answer_short": "momentum = mass x velocity",
     "correctness": "partial", "grounded": "yes", "refusal": "na"},
    {"question": "What is Newton's third law of motion?",
     "answer_short": "Equal and opposite forces",
     "correctness": "correct", "grounded": "no", "refusal": "na"},
    {"question": "What happens when unbalanced force acts?",
     "answer_short": "Object brought into motion",
     "correctness": "correct", "grounded": "yes", "refusal": "na"},
    {"question": "What is SI unit of force?",
     "answer_short": "kg m s-2 or newton N",
     "correctness": "correct", "grounded": "yes", "refusal": "na"},
    {"question": "Relationship between force mass acceleration?",
     "answer_short": "Refused - not found",
     "correctness": "wrong", "grounded": "na", "refusal": "na"},
    {"question": "What is law of conservation of momentum?",
     "answer_short": "Refused - not found",
     "correctness": "wrong", "grounded": "na", "refusal": "na"},
    {"question": "What happens to body at rest with no force?",
     "answer_short": "Remains at rest due to inertia",
     "correctness": "correct", "grounded": "no", "refusal": "na"},
    {"question": "No external force on moving object?",
     "answer_short": "Continues with same velocity",
     "correctness": "correct", "grounded": "no", "refusal": "na"},
    {"question": "How does mass affect motion?",
     "answer_short": "Refused - not found",
     "correctness": "wrong", "grounded": "na", "refusal": "na"},
    {"question": "Why passengers fall forward when bus brakes?",
     "answer_short": "Inertia - body continues in same state",
     "correctness": "correct", "grounded": "no", "refusal": "na"},
    {"question": "Who is Prime Minister of India?",
     "answer_short": "Refused correctly",
     "correctness": "na", "grounded": "na", "refusal": "yes"},
    {"question": "What is photosynthesis?",
     "answer_short": "Refused correctly",
     "correctness": "na", "grounded": "na", "refusal": "yes"},
    {"question": "What is capital of France?",
     "answer_short": "Refused correctly",
     "correctness": "na", "grounded": "na", "refusal": "yes"},
    {"question": "Explain quantum entanglement?",
     "answer_short": "Refused correctly",
     "correctness": "na", "grounded": "na", "refusal": "yes"},
]

# calculate scores
in_scope = [e for e in eval_scored if e['correctness'] != 'na']
correct = [e for e in in_scope if e['correctness'] in ['correct', 'partial']]
grounded = [e for e in in_scope if e['grounded'] == 'yes']
out_scope = [e for e in eval_scored if e['refusal'] != 'na']
refused = [e for e in out_scope if e['refusal'] == 'yes']

print("=== FINAL EVALUATION RESULTS ===")
print(f"Total questions:      {len(eval_scored)}")
print(f"In-scope:             {len(in_scope)}")
print(f"Correct:              {len(correct)}/{len(in_scope)} ({len(correct)/len(in_scope)*100:.0f}%)")
print(f"Grounded:             {len(grounded)}/{len(in_scope)} ({len(grounded)/len(in_scope)*100:.0f}%)")
print(f"Appropriate refusals: {len(refused)}/{len(out_scope)} ({len(refused)/len(out_scope)*100:.0f}%)")

print("\n=== 3 WORKING EXAMPLES ===")
working = [e for e in eval_scored if e['correctness'] == 'correct' and e['grounded'] == 'yes']
for e in working[:3]:
    print(f"✅ {e['question']} → {e['answer_short']}")

print("\n=== 2 FAILURE CASES ===")
failures = [e for e in eval_scored if e['correctness'] == 'wrong']
for e in failures[:2]:
    print(f"❌ {e['question']} → {e['answer_short']}")

# save to csv
with open('evaluation_results.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=[
        'question', 'answer_short', 'correctness', 'grounded', 'refusal'])
    writer.writeheader()
    writer.writerows(eval_scored)

print("\nevaluation_results.csv saved!")

=== FINAL EVALUATION RESULTS ===
Total questions:      17
In-scope:             13
Correct:              10/13 (77%)
Grounded:             4/13 (31%)
Appropriate refusals: 4/4 (100%)

=== 3 WORKING EXAMPLES ===
✅ What is Newton's first law of motion? → Law of inertia - objects resist change in motion
✅ What happens when unbalanced force acts? → Object brought into motion
✅ What is SI unit of force? → kg m s-2 or newton N

=== 2 FAILURE CASES ===
❌ Relationship between force mass acceleration? → Refused - not found
❌ What is law of conservation of momentum? → Refused - not found

evaluation_results.csv saved!
